# 第 7 章 · MoE 混合专家

> 📄 配套动画网页：`../07_moe.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第二部分 · 模型

MoE 把前馈层拆成多个「专家」，每个 token 只激活其中一两个——参数量上去了，计算量却几乎不变。

---


## 🎯 本章目标

- 理解稀疏激活：用参数换算力
- 掌握 `torch.topk`、`F.one_hot`、`index_add_`
- 理解负载均衡辅助损失的作用
- 能从零实现 `MOEFeedForward`

---


## 📖 背景：稀疏激活

```python
scores   = softmax(router(x))          # 每个 token 对 N 个专家打分
topk_w, topk_i = topk(scores, k=1)     # 只选分数最高的 k 个
y = Σ weight_i * expert_i(x)           # 只调用被选中的专家
```

稠密模型每个 token 用全部参数；MoE 每个 token 只用一小部分。
所以 MoE 的**总参数量**可以是稠密的 3 倍，但**单 token 激活参数**几乎一样。

问题也随之而来：如果所有 token 都涌向同一个专家，其它专家就「饿死」了。
所以需要**负载均衡辅助损失**。

---



## 🖼 官方配图

![MiniMind MoE 结构：前馈层被替换为多个专家 + 一个路由器（Qwen3-MoE 风格，无共享专家）](../images/LLM-structure-moe.jpg)

*MiniMind MoE 结构：前馈层被替换为多个专家 + 一个路由器（Qwen3-MoE 风格，无共享专家）*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 7.1 `torch.topk`：取分数最高的 k 个

In [ ]:
import torch

scores = torch.tensor([[0.1, 0.6, 0.2, 0.1],
                       [0.4, 0.1, 0.4, 0.1]])
w, idx = torch.topk(scores, k=1, dim=-1, sorted=False)
print("top1 权重:", w.tolist())
print("top1 索引:", idx.tolist())

w2, idx2 = torch.topk(scores, k=2, dim=-1, sorted=True)
print("top2 索引:", idx2.tolist())

### 7.2 `F.one_hot`：把索引变成独热

用来统计「每个专家被选了多少次」。

In [ ]:
import torch.nn.functional as F

idx = torch.tensor([[0], [2], [1], [0]])       # 4 个 token 选了哪个专家
oh = F.one_hot(idx, num_classes=4).float()     # [4, 1, 4]
print("one_hot 形状:", oh.shape)
load = oh.mean(0)                              # 每个专家被选中的比例
print("负载分布:", load.squeeze().tolist())

### 7.3 `index_add_`：按需累加

MoE 的循环里，我们只对「选中了当前专家」的那些 token 做计算，再把结果累加回原位置。

In [ ]:
y = torch.zeros(4, 3)                    # 4 个 token，每个 3 维
token_idx = torch.tensor([0, 3])         # 这两个 token 属于本专家
vals = torch.ones(2, 3) * 5.0
y.index_add_(0, token_idx, vals)         # 按索引累加
print(y)

### 7.4 布尔掩码分派

`topk_idx == i` 得到一个布尔张量，标出哪些 token 选了专家 i。

In [ ]:
topk_idx = torch.tensor([[0], [2], [0], [1]])
i = 0
mask = (topk_idx == i)
print("mask 形状:", mask.shape)
print("哪些 token 选了专家 0:", mask.any(dim=-1).nonzero().flatten().tolist())

### 7.5 前置依赖：`FeedForward`

MoE 的「专家」就是一个前馈层，也就是第 6 章你写过的 `FeedForward`。
下面给一份**等价的参考实现**——不是让你抄，而是让本章可以独立测试。
（如果你第 6 章已经写好了，可以直接换成你自己的版本。）

In [ ]:
from torch import nn

class FeedForward(nn.Module):
    """参考实现，等价于第 6 章的 SwiGLU 前馈层"""
    def __init__(self, config, intermediate_size=None):
        super().__init__()
        d = intermediate_size or config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, d, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, d, bias=False)
        self.down_proj = nn.Linear(d, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

MoE 的实现里有一个容易被忽略的细节：反向传播的梯度路径。请仔细看提示。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 7.6 实现 `MOEFeedForward`

**要求**：
1. `self.gate = nn.Linear(hidden_size, num_experts, bias=False)`（路由器）
2. `self.experts = nn.ModuleList([FeedForward(config) for _ in range(num_experts)])`
3. `forward(x)`：
   - 把 `[B, S, d]` 展平成 `[B*S, d]`
   - 算 router 分数 → softmax → `topk`
   - 对每个专家：找出选了它的 token，算输出并**按权重累加**回 `y`
   - reshape 回 `[B, S, d]`
4. 训练时计算 `aux_loss`（负载均衡），否则返回 0

**关键细节**：MiniMind 在 `num_experts_per_tok == 1` 时用「直通估计」——
前向让权重恒为 1.0，但用 `top1 - top1.detach() + 1.0` 保留梯度路径。
这样单专家路由时权重不会退化。

In [ ]:
# ======== 你的实现：MOEFeedForward ========
#
# 【要求】
#   - 继承 nn.Module
#   - gate = nn.Linear(hidden_size, num_experts, bias=False)
#   - experts = nn.ModuleList([FeedForward(config, moe_intermediate_size) ...])
#   - forward: 展平 -> softmax(gate(x)) -> topk -> 逐专家 index_add_ 累加
#   - k==1 时权重用 top1 - top1.detach() + 1.0 保持梯度
#   - 训练时计算 aux_loss = (load * prob).sum() * num_experts * coef
#
# 【可用到的函数 / 方法】
#   - `torch.topk`、`F.softmax`
#   - `torch.zeros_like`、`Tensor.index_add_`
#   - `F.one_hot`、`Tensor.nonzero`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：MOEFeedForward ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)

    class Cfg:
        hidden_size = 16
        intermediate_size = 32
        moe_intermediate_size = 32
        num_experts = 4
        num_experts_per_tok = 1
        norm_topk_prob = True
        router_aux_loss_coef = 5e-4

    moe = MOEFeedForward(Cfg())
    x = torch.randn(2, 6, 16)
    y = moe(x)
    assert tuple(y.shape) == (2, 6, 16), f"输出形状应为 (2, 6, 16)，得到 {tuple(y.shape)}"
    assert torch.isfinite(y).all(), "输出出现了 inf/nan"

    xf, yf = x.reshape(-1, 16), y.reshape(-1, 16)
    top1 = F.softmax(moe.gate(xf), dim=-1).argmax(-1)
    for i in range(xf.shape[0]):
        e = top1[i].item()
        close(yf[i], moe.experts[e](xf[i]), tol=1e-4,
              msg=f"token {i} 选了专家 {e}，输出应等于该专家的输出（k=1 时权重为 1）")

check("MOEFeedForward 形状 / 单专家路由", _t)

### 7.7 实现辅助损失 `aux_loss`

**要求**：输入 router 分数和 topk 索引，返回负载均衡损失。

```python
load = one_hot(topk_idx, num_experts).float().mean(0)   # 实际负载
prob = scores.mean(0)                                    # 平均路由概率
aux  = (load * prob).sum() * num_experts * coef
```

**直觉**：当负载完全均匀时，`load ≈ prob ≈ 1/N`，`sum = N·(1/N)² = 1/N`，
乘 `N` 后得到 1。所以 aux_loss 越接近 1 表示越均衡。

In [ ]:
# ======== 你的实现：aux_loss ========
#
# 【要求】
#   - 输入：scores [B*S, N]、topk_idx [B*S, k]、num_experts、coef
#   - load = F.one_hot(topk_idx, N).float().mean(0)
#   - prob = scores.mean(0)
#   - 返回 (load * prob).sum() * num_experts * coef
#
# 【可用到的函数 / 方法】
#   - `F.one_hot`
#   - `Tensor.mean`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：aux_loss ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    balanced = torch.tensor([[0.7, 0.1, 0.1, 0.1],
                             [0.1, 0.7, 0.1, 0.1],
                             [0.1, 0.1, 0.7, 0.1],
                             [0.1, 0.1, 0.1, 0.7]])
    idx_b = balanced.argmax(-1, keepdim=True)
    loss_b = aux_loss(balanced, idx_b, num_experts=4, coef=1.0)
    close(loss_b.item(), 1.0, tol=1e-5, msg="负载与概率都均匀时，aux_loss 应为 1.0")

    skewed = torch.tensor([[0.9, 0.03, 0.03, 0.04]] * 4)
    idx_s = skewed.argmax(-1, keepdim=True)
    loss_s = aux_loss(skewed, idx_s, num_experts=4, coef=1.0)
    assert loss_s.item() > loss_b.item(), \
        f"负载集中时 aux_loss 应更大（这正是它要惩罚的），得到 {loss_s.item():.3f}"

check("aux_loss 负载越均匀值越接近 1", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能解释 MoE 为什么「参数多但算力省」
- [ ] 负载均衡损失接近 1 意味着什么？
- [ ] 你的实现里被选中的 token 输出正确累加了吗？
- [ ] k=1 时的直通估计你理解了吗？

**思考题**：如果路由器对所有 token 都给出相同分数，会发生什么？这时 aux_loss 是多少？
